## IMMUN 306QC Python Bootcamp / Session 2
# Data wrangling with Pandas
*Ralph Estanboulieh (2026)*

*This notebook was created **without** the use of AI.*

The first session of this bootcamp, which covered Python's basics, culminated in a brief guided tour of some of Python's most basic modules, including `math`, `statistics`, and `random`. Now, we get to explore some of its most important modules for data processing, **Pandas**, and plotting, **Matplotlib**.

## Pandas

Welcome to the zoo! Pandas (indeed, in the plural form), is Python's bread-and-butter data exploration and wrangling module. Pandas can deal with all common types of tabular data: `CSV` (comma-separated values), `tsv` (tab-separated values), XLS (Excel spreadsheets), Parquet, HDF5, and so many others. Once data is read by Pandas, we can do so much with it: we can subset and manipulate rows and columns in all sorts of ways, we can perform new calculations, we can plot data, we can join tables and flip them and restructure them however we desire.

One of the main goals of this course is to teach you how to use Python's ScanPy module to do sequencing analysis — that will happen after this bootcamp. While ScanPy uses a *different* module to handle large tabular data called AnnData (stands for "annotated data"), we still believe it important to teach you about Pandas because it is a more generalizable framework that will prove useful in many more contexts than single-cell RNA sequencing analysis.

While this session will cover Panda's basics, I highly recommend perusing Pandas' very helpful documentation, which can be found here: https://pandas.pydata.org/docs/. For the visual learners, I will occasionally use some of Pandas' own diagrams throughout this workshop.

## Data frames and series

Pandas is centered around the very intuitive `DataFrame` object: 

<center><img src="figures/01_table_dataframe.svg" width=500 align="center"></center>

Rows are rows, and columns are columns. Columns have names — labels — and rows have names too — indexes. Pandas DataFrames support most reasonable data types: numbers, text, categorical data (which I will explain later), and more. 

Selecting one DataFrame column (which we will do in a minute) returns a Series object:

<center><img src="figures/01_table_series.svg" width=200/></center>

First, let's look at some basic ways to create Series and DataFrames, just to understand how they work. But before anything, let's import pandas.

In [2]:
import pandas as pd
print(pd.__version__) # version should be 3.0.5

3.0.5


The `as pd` part allows us to invoke Pandas functions as `pd.function_name` rather than `pandas.function_name`. Yes, these four extra letters make a big difference. 

## `Series`

Creating `Series` and `DataFrame` objects is very intuitive: we just need some form of structured data, like lists or dictionaries:

In [32]:
s = pd.Series([10,20,30,40,50]) # easy!
s

0    10
1    20
2    30
3    40
4    50
dtype: int64

We can name the rows with the `index` argument and name the entire Series with the `name` argument — and we can even get away with a single number as data (see how it gets propagated!):

In [248]:
s = pd.Series(50, index=['A', 'B', 'C', 'D', 'E'], name="example")
s

A    50
B    50
C    50
D    50
E    50
Name: example, dtype: int64

Notice the `dtype` at the bottom of the series? That's the data type. All Series, as well as each column in a DataFrame, have a data type. This helps Pandas stay organized and optimize its computations. In this case, `dtype: int64` means the data type is a 64-bit integer (integers that are stored with 64 0s and 1s in the computer's memory). Other dtypes include `float` for decimals, `str` for strings, `bool` for bools (`True` or `False`), and even `complex` for complex numbers!

Another way to create a Series is with a Python dictionary in which the keys become the row indexes and the values become the Series data:

In [249]:
s = pd.Series({'A': 10, 'B': 20, 'C': 30, 'D': 40, 'E': 50})
s

A    10
B    20
C    30
D    40
E    50
dtype: int64

Checking the size of a Series is easy:

In [250]:
s.shape

(5,)

Here, `shape` is called an **attribute**. Indeed, not everything that comes after dot notation is necessarily a function (aka a method); objects usually have both methods (functions) and attributes (variables which store data)

Subsetting Series objects is also intuitive:

In [251]:
print(s['A'])

10


Another way to subset series (and DataFrames) is with the **index locator** method, which works exactly like subsetting a basic Python list:

In [252]:
print(s.iloc[:3])
print(s.iloc[2:])
print(s.iloc[2:4])

A    10
B    20
C    30
dtype: int64
C    30
D    40
E    50
dtype: int64
C    30
D    40
dtype: int64


We can also change the values of a Series by reassignment

In [253]:
s['A'] = 5
s

A     5
B    20
C    30
D    40
E    50
dtype: int64

What if we wanted to select only the rows with values larger than 20? A very cool thing about Series and DataFrames is that they can be intuitively used with operators like `<`, `>`, `==`, and other comparison operators. This will return a series of boolean values (`True` or `False`): 

In [254]:
s > 20

A    False
B    False
C     True
D     True
E     True
dtype: bool

... which we can use to subset our series (with the square brackets)!

In [255]:
s[s>20]

C    30
D    40
E    50
dtype: int64

Doing math with Series is very intuitive — there is usually no need to loop through values

In [256]:
((s * 2) + 20) / 10

A     3.0
B     6.0
C     8.0
D    10.0
E    12.0
dtype: float64

You can even raise it to the power of your liking:

In [257]:
s ** 2

A      25
B     400
C     900
D    1600
E    2500
dtype: int64

There is of course a lot more math you can do with them, but that requires knowledge of another advanced math module while we will cover in the next session.

The operations above are called **vectorized operations**, meaning that these operations are done ultra-efficiently with code that is designed to manipulate objects as vectors. When possible, always opt for a vectorized version of your algorithm (and if you don't know if one exists, look it up!)

And of course, Series and DataFrames can contain data of different types, like integers and strings in the example below:

In [258]:
s = pd.Series([10, 20, 30, 40, 'Hello!'], index=['A', 'B', 'C', 'D', 'E'], name="example")
s

A        10
B        20
C        30
D        40
E    Hello!
Name: example, dtype: object

Notice the `dtype` is now `object`, which is Pandas' way to codifying data of variable types. In this (pretty rare and unuseful) case, doing math is more complicated than directly using `+` and `-` with the series — but ideally you should avoid these scenarios in the first place.

## `DataFrame`

A DataFrame is a 2-dimensional version of a series, or simply, a table with labeled rows and named columns. In fact, each column of a DataFrame is a series.

There are many ways to create a DataFrame. The most common way is to probably read a CSV or Excel file using the functions we will learn about later, but if you wish to create a DataFrame from scratch and without external data, one way to do so is also with a dictionary:

In [259]:
d = pd.DataFrame({'cell_type': ['Th', 'Tc', 'Treg', 'B', 'NK', 'DC'], 
                  'count': [123, 234, 345, 456, 567, 678]},
                index = ['cell1', 'cell2', 'cell3', 'cell4','cell5', 'cell6'])
d

,cell_type,count
cell1,Th,123
cell2,Tc,234
cell3,Treg,345
cell4,B,456
cell5,NK,567
cell6,DC,678


*(Notice how for better readability, I used multiple lines for the dictionary and `index` argument. Python doesn't care about jumping lines in this context, so use it to your advantage!)*

Beautiful! We can check the size in at least two ways:

In [260]:
print(d.shape) # n_rows, n_cols
print(d.size) # total number of elements (n_rows * n_cols)

(6, 2)
12


It is also easy to change the index of a DataFrame. Simply, we reassign the `index` attribute (remember? the data-holding variables inside Python objects):

In [261]:
d.index # see the data held in this attribute

Index(['cell1', 'cell2', 'cell3', 'cell4', 'cell5', 'cell6'], dtype='str')

In [262]:
d.index = ['cluster1', 'cluster2', 'cluster3', 'cluster4', 'cluster5', 'cluster6']
d

,cell_type,count
cluster1,Th,123
cluster2,Tc,234
cluster3,Treg,345
cluster4,B,456
cluster5,NK,567
cluster6,DC,678


Renaming columns is also straightforward, but since there can be many to rename, we have to specify which ones and what the new names should be, just like the find-and-replace feature of most text editors. To do this, we use the `rename` method, which does not change the original dataframe, but instead returns a copy of it with the renaming. To make that change permanent, we can assign that output to a new variable (or the one with the un-renamed DataFrame):

In [263]:
d = d.rename(columns={'count': 'cell_count'})
d

,cell_type,cell_count
cluster1,Th,123
cluster2,Tc,234
cluster3,Treg,345
cluster4,B,456
cluster5,NK,567
cluster6,DC,678


Amazingly, the `rename` function can also rename indexes the same way:

In [264]:
d.rename(index={'cluster3': 'C3', 'cluster6':'C6'})

,cell_type,cell_count
cluster1,Th,123
cluster2,Tc,234
C3,Treg,345
cluster4,B,456
cluster5,NK,567
C6,DC,678


A useful pair of functions to view a DataFrame are `.head()` and `.tail()`, which display the first and last 5 rows, respectively, but you can also specifcy how many rows you wish to see!

In [265]:
d.head()

,cell_type,cell_count
cluster1,Th,123
cluster2,Tc,234
cluster3,Treg,345
cluster4,B,456
cluster5,NK,567


In [266]:
d.head(3)

,cell_type,cell_count
cluster1,Th,123
cluster2,Tc,234
cluster3,Treg,345


In [267]:
d.tail()

,cell_type,cell_count
cluster2,Tc,234
cluster3,Treg,345
cluster4,B,456
cluster5,NK,567
cluster6,DC,678


In [268]:
d.tail(2)

,cell_type,cell_count
cluster5,NK,567
cluster6,DC,678


## Setting and resetting the index

What if we wanted a different column to become the index? This can be useful in case you wanted to access data quickly using that column. Enter `set_index()`, which conveniently does exactly that!

In [269]:
d.set_index('cell_type')

,cell_count
cell_type,
Th,123
Tc,234
Treg,345
B,456
NK,567
DC,678


Keep in mind that `set_index()` does not change the original DataFrame, but instead returns a copy. So to make the change permanent, you would have to assign the copy to the DataFrame itself (something like `d = d.set_index(...)`).

Now what if we wanted to turn the index into a normal column. Pandas offers a convenient way to do so: `reset_index()`, which turns the index into a column and creates a new simple integer index.

In [270]:
d.reset_index()

,index,cell_type,cell_count
0,cluster1,Th,123
1,cluster2,Tc,234
2,cluster3,Treg,345
3,cluster4,B,456
4,cluster5,NK,567
5,cluster6,DC,678


Interestingly enough, calling `reset_index()` on a Series returns a DataFrame! (unless `drop=True`)

In [55]:
s = pd.Series({'A': 10, 'B': 20, 'C': 30, 'D': 40, 'E': 50}, name='Number')
s.index.name = 'myIndex' # try commenting this line out
s

myIndex
A    10
B    20
C    30
D    40
E    50
Name: Number, dtype: int64

In [56]:
s.reset_index(drop=False) # try drop=True and try name="<whatever you want>"

,myIndex,Number
0,A,10
1,B,20
2,C,30
3,D,40
4,E,50


To rename that new column, you can either use `rename()` as seen above, or use the argument `names` in `reset_index()`:

In [271]:
d.reset_index(names='cluster_name')

,cluster_name,cell_type,cell_count
0,cluster1,Th,123
1,cluster2,Tc,234
2,cluster3,Treg,345
3,cluster4,B,456
4,cluster5,NK,567
5,cluster6,DC,678


(The reason it is `names` in the plural form is because Pandas supports a multi-level index (and multi-level columns too!), so many index columns can be reset and renamed at once.)

Notice how `set_index()` removes the original index column, one nifty way to keep the old index as a column is to use `reset_index()` first to turn the old index into a column and then use `set_index()` to make the new index:

In [272]:
d.reset_index(names='cluster_name').set_index('cell_type')

,cluster_name,cell_count
cell_type,,
Th,cluster1,123
Tc,cluster2,234
Treg,cluster3,345
B,cluster4,456
NK,cluster5,567
DC,cluster6,678


## Broadcasting operations

Just like with series, we can use many operators with DataFrames to do math, check inequalities, and even concatenate strings!

But first, let's select a column in the DataFrame:

In [273]:
d['cell_count']

cluster1    123
cluster2    234
cluster3    345
cluster4    456
cluster5    567
cluster6    678
Name: cell_count, dtype: int64

Notice that a Series is returned! Let's try to do some simple math

In [274]:
d['cell_count'] + 10000 

cluster1    10123
cluster2    10234
cluster3    10345
cluster4    10456
cluster5    10567
cluster6    10678
Name: cell_count, dtype: int64

Or check an inequality:

In [275]:
d['cell_count'] > 400

cluster1    False
cluster2    False
cluster3    False
cluster4     True
cluster5     True
cluster6     True
Name: cell_count, dtype: bool

Or even concatenate strings:

In [276]:
d['cell_type'] + '_01'

cluster1      Th_01
cluster2      Tc_01
cluster3    Treg_01
cluster4       B_01
cluster5      NK_01
cluster6      DC_01
Name: cell_type, dtype: str

Pandas and other common data anlysis modules have this very intuitive and convenient feature called **broadcasting**, which simply means that an operation is "broadcastedd" across the array or Series or DataFrame in a very efficient manner. These operations are also called **vectorized** operations, meaning that they are efficiently applied across vectors and matrices using C instead of Python. (C is a low-level but extremely fast programming language!)

## Basic summary statistics

Now that we have some (made-up) data, let's try to `describe` it: 

In [277]:
d['cell_count'].describe()

count      6.000000
mean     400.500000
std      207.661985
min      123.000000
25%      261.750000
50%      400.500000
75%      539.250000
max      678.000000
Name: cell_count, dtype: float64

The `describe()` function calculates a set of useful summary statitics. If we wanted only one of them:

In [278]:
print(d['cell_count'].count())
print(d['cell_count'].mean())
print(d['cell_count'].max())
print(d['cell_count'].min())

6
400.5
678
123


But summary statistics are not limited to the above:

In [279]:
print(d['cell_count'].var()) # variance
print(d['cell_count'].median()) # median
print(d['cell_count'].skew()) # skew  (0 because the distribution is perfectly centered)
print(d['cell_count'].sum())
print(d['cell_count'].prod()) # the product of all values, which is HUGE!

43123.5
400.5
0.0
2403
1740674869446240


These are only aggregate satistics, but we will later explore more complex operations. A comprehensive list of these methods can be found here: https://pandas.pydata.org/docs/reference/frame.html#api-dataframe-stats

The aggregate statistics seen above (and others not shown) do not only apply to columns, but also rows! To do so, we can use the `axis=1` argument. The default value of axis, `axis=0`, applies the operations along the row axis (e.g. sums all the rows in a column, so the stats are per column), whereas `axis=1` applies the operations along the columns axis (e.g. sums all the coluns in a row, so the stats are per row). This is a very important feature of these operations and is a standard that applies to other functions and even related Python libraries.

In [280]:
d2 = pd.DataFrame(
    {'gene1': [1,2,3,4,5], 'gene2': [1,2,3,4,5], 'gene3': [1,2,3,4,5]}
)
d2

,gene1,gene2,gene3
0,1,1,1
1,2,2,2
2,3,3,3
3,4,4,4
4,5,5,5


In [281]:
d2.sum(axis=0)

gene1    15
gene2    15
gene3    15
dtype: int64

In [282]:
d2.sum(axis=1)

0     3
1     6
2     9
3    12
4    15
dtype: int64

## Column addition and deletion

Creating new columns based on existing ones is also easy: simply assign a new columns to some values!

In [283]:
d['is_count_large'] = d['cell_count'] > 400
d

,cell_type,cell_count,is_count_large
cluster1,Th,123,False
cluster2,Tc,234,False
cluster3,Treg,345,False
cluster4,B,456,True
cluster5,NK,567,True
cluster6,DC,678,True


By default, new columns are always added at the very end (to the right), but `DataFrame.insert()` lets us specify where to add the colum. The first argument is where to place the new column (0 for 1st column, 1 for 2nd, 2 for 3rd, etc.), the second argument is the name of the new column (`new_col` in this case), and the third argument is the value of the new column, which could be anything you want!

In [284]:
d.insert(2, "Cell %", 100 * d['cell_count']/d['cell_count'].sum()) 
d

,cell_type,cell_count,Cell %,is_count_large
cluster1,Th,123,5.118602,False
cluster2,Tc,234,9.737828,False
cluster3,Treg,345,14.357054,False
cluster4,B,456,18.976280,True
cluster5,NK,567,23.595506,True
cluster6,DC,678,28.214732,True


Let's create a column so we can delete it later:

In [285]:
d['useless_col'] = 20
d

,cell_type,cell_count,Cell %,is_count_large,useless_col
cluster1,Th,123,5.118602,False,20
cluster2,Tc,234,9.737828,False,20
cluster3,Treg,345,14.357054,False,20
cluster4,B,456,18.976280,True,20
cluster5,NK,567,23.595506,True,20
cluster6,DC,678,28.214732,True,20


*(Notice how the value 20 is copied to all the rows of the DataFrame? This is also broadcasting!)*

Now let's delete what we created:

In [286]:
del d['useless_col']
d

,cell_type,cell_count,Cell %,is_count_large
cluster1,Th,123,5.118602,False
cluster2,Tc,234,9.737828,False
cluster3,Treg,345,14.357054,False
cluster4,B,456,18.976280,True
cluster5,NK,567,23.595506,True
cluster6,DC,678,28.214732,True


## Slicing and subsetting

There are many useful ways to select, slice, and subset a DataFrame. We've already seen the most basic way to select a column:

In [287]:
d['cell_type']

cluster1      Th
cluster2      Tc
cluster3    Treg
cluster4       B
cluster5      NK
cluster6      DC
Name: cell_type, dtype: str

Or even multiple columns (notice the double brackets!):

In [288]:
d[['cell_type', 'cell_count']]

,cell_type,cell_count
cluster1,Th,123
cluster2,Tc,234
cluster3,Treg,345
cluster4,B,456
cluster5,NK,567
cluster6,DC,678


An equally basic way to select rows:

In [289]:
d[2:4]

,cell_type,cell_count,Cell %,is_count_large
cluster3,Treg,345,14.357054,False
cluster4,B,456,18.976280,True


To select based on row index and column name, use the `DataFrame.loc[rows, columns]` method:

In [290]:
d.loc['cluster1', 'cell_type']

'Th'

In [291]:
d.loc['cluster1', :] # row with index cluster1, all columns (:)

cell_type               Th
cell_count             123
Cell %            5.118602
is_count_large       False
Name: cluster1, dtype: object

In [292]:
d.loc[:, 'cell_count'] # column with name cell_count, all rows (:)

cluster1    123
cluster2    234
cluster3    345
cluster4    456
cluster5    567
cluster6    678
Name: cell_count, dtype: int64

`DataFrame.loc[rows, cols]` can also take a list of rows/columns-of-interest:

In [293]:
d.loc[['cluster1', 'cluster5'], :] 

,cell_type,cell_count,Cell %,is_count_large
cluster1,Th,123,5.118602,False
cluster5,NK,567,23.595506,True


In [294]:
d.loc[:, ['cell_count', 'Cell %']]

,cell_count,Cell %
cluster1,123,5.118602
cluster2,234,9.737828
cluster3,345,14.357054
cluster4,456,18.976280
cluster5,567,23.595506
cluster6,678,28.214732


In [295]:
d.loc[['cluster1', 'cluster5'], ['cell_count', 'Cell %']]

,cell_count,Cell %
cluster1,123,5.118602
cluster5,567,23.595506


If you prefer to deal with indexes rather than row and column names, you can use the `DataFrame.iloc[row_index, col_index]` method:

In [296]:
print(d.iloc[2, 1])

345


In [297]:
d.iloc[3:5, 2:5]

,Cell %,is_count_large
cluster4,18.976280,True
cluster5,23.595506,True


In [298]:
d.iloc[[0,2,4], :]

,cell_type,cell_count,Cell %,is_count_large
cluster1,Th,123,5.118602,False
cluster3,Treg,345,14.357054,False
cluster5,NK,567,23.595506,True


Finally, just like Series, DataFrames can be subsetted with arrays of bools (`True` and `False`); of course, the number of bools must match the number of rows in the DataFrame. Let's use one of the boolean columns we already have in the DataFrame:

In [299]:
d['is_count_large']

cluster1    False
cluster2    False
cluster3    False
cluster4     True
cluster5     True
cluster6     True
Name: is_count_large, dtype: bool

In [300]:
d[d['is_count_large']]

,cell_type,cell_count,Cell %,is_count_large
cluster4,B,456,18.976280,True
cluster5,NK,567,23.595506,True
cluster6,DC,678,28.214732,True


But we don't have to restrict ourselves to existing columns, we can generate boolean series on the go:

In [301]:
d[d['cell_count'] > 400]

,cell_type,cell_count,Cell %,is_count_large
cluster4,B,456,18.976280,True
cluster5,NK,567,23.595506,True
cluster6,DC,678,28.214732,True


The expression inside the brackets (`d['cell_count'] > 400`) silently returns a Series of boolean values which are instantly used to subset the rows of the DataFrame. Magic!

Now say we wanted to look at the cell counts from that selection, we would simply select the column after making the selection:

In [302]:
d[d['cell_count'] > 400]['cell_count']

cluster4    456
cluster5    567
cluster6    678
Name: cell_count, dtype: int64

What if we wanted to select rows based on whether a specific value is in a set of values of interest? Consider the `Series.isin()` method (where the Series could be a column in a DataFrame), which takes a list of values of interest and returns a boolean series indicating whether each value in the series belongs to that list:

In [303]:
d['cell_type'].isin(['Treg', 'B', 'DC'])

cluster1    False
cluster2    False
cluster3     True
cluster4     True
cluster5    False
cluster6     True
Name: cell_type, dtype: bool

Now take that code and put it between brackets to locate the rows in the DataFrame:

In [304]:
d[d['cell_type'].isin(['Treg', 'B', 'DC'])]

,cell_type,cell_count,Cell %,is_count_large
cluster3,Treg,345,14.357054,False
cluster4,B,456,18.976280,True
cluster6,DC,678,28.214732,True


## Transposing

To transpose a DataFrame, meaning to flip the columns and rows, we can either use the `DataFrame.T` attribute or the `DataFrame.transpose()` method. 

In [305]:
d.T

,cluster1,cluster2,cluster3,cluster4,cluster5,cluster6
cell_type,Th,Tc,Treg,B,NK,DC
cell_count,123,234,345,456,567,678
Cell %,5.118602,9.737828,14.357054,18.97628,23.595506,28.214732
is_count_large,False,False,False,True,True,True


In [306]:
d.transpose()

,cluster1,cluster2,cluster3,cluster4,cluster5,cluster6
cell_type,Th,Tc,Treg,B,NK,DC
cell_count,123,234,345,456,567,678
Cell %,5.118602,9.737828,14.357054,18.97628,23.595506,28.214732
is_count_large,False,False,False,True,True,True


## Sorting

You can sort a Series and DataFrame by either indexes (row labels) using `sort_index()`, or column values using `sort_values()` (which requires a column name to sort by).

Since most our columns are already sorted in ascending order, let's sort them in descending order. The way to do this is to set the `ascending` argument to `False`.

In [307]:
d.sort_index(ascending=False)

,cell_type,cell_count,Cell %,is_count_large
cluster6,DC,678,28.214732,True
cluster5,NK,567,23.595506,True
cluster4,B,456,18.976280,True
cluster3,Treg,345,14.357054,False
cluster2,Tc,234,9.737828,False
cluster1,Th,123,5.118602,False


In [308]:
d.sort_values(by='cell_count', ascending=False)

,cell_type,cell_count,Cell %,is_count_large
cluster6,DC,678,28.214732,True
cluster5,NK,567,23.595506,True
cluster4,B,456,18.976280,True
cluster3,Treg,345,14.357054,False
cluster2,Tc,234,9.737828,False
cluster1,Th,123,5.118602,False


Of course, we can also sort text:

In [309]:
d.sort_values(by='cell_type')

,cell_type,cell_count,Cell %,is_count_large
cluster4,B,456,18.976280,True
cluster6,DC,678,28.214732,True
cluster5,NK,567,23.595506,True
cluster2,Tc,234,9.737828,False
cluster1,Th,123,5.118602,False
cluster3,Treg,345,14.357054,False


We can also sort using multiple columns! To demonstrate this, let's make a simpler DataFrame:

In [310]:
df_sort = pd.DataFrame({
    "col1": [3,3,3,3,3,3,3,3,3,2,2,2,2,2,2,2,2,2,1,1,1,1,1,1,1,1,1],
    "col2": [3,2,1,3,2,1,3,2,1,3,2,1,3,2,1,3,2,1,3,2,1,3,2,1,3,2,1],
    "col3": list("CCCBBBAAACCCBBBAAACCCBBBAAA")
})

df_sort

,col1,col2,col3
0,3,3,C
1,3,2,C
2,3,1,C
3,3,3,B
4,3,2,B
5,3,1,B
6,3,3,A
7,3,2,A
8,3,1,A
9,2,3,C


Sorting by `col1` only will not guarantee that `col2` and `col3` are also sorted:

In [311]:
df_sort.sort_values(['col1'])

,col1,col2,col3
24,1,3,A
25,1,2,A
23,1,1,B
22,1,2,B
20,1,1,C
21,1,3,B
18,1,3,C
26,1,1,A
19,1,2,C
12,2,3,B


Sorting by both `col1` and `col2` (in that specific order) means that `col1` is sorted first and `col2` is sorted next **without** breaking the sort of `col1`

In [312]:
df_sort.sort_values(['col1', 'col2']) # try switching the order of the columns!

,col1,col2,col3
20,1,1,C
23,1,1,B
26,1,1,A
19,1,2,C
22,1,2,B
25,1,2,A
18,1,3,C
21,1,3,B
24,1,3,A
11,2,1,C


So let's sort by all three!

In [313]:
df_sort.sort_values(['col1', 'col2', 'col3']) # try switching the order of the columns!

,col1,col2,col3
26,1,1,A
23,1,1,B
20,1,1,C
25,1,2,A
22,1,2,B
19,1,2,C
24,1,3,A
21,1,3,B
18,1,3,C
17,2,1,A


There you go! Intuitively, the first column in the list is given the priority, and each column that comes next is sorted without breaking the sorting of the previous columns.

## String methods

Say we wanted to do some more advanced work with strings. For instance, say we want to create a new column that says whether a cell type is a T cell subpopulation or not. One way to do this is by using a for loop:  

In [314]:
T_subpop = [] # initiate empty list
for x in d['cell_type']:
    print(x)
    if x[0] == "T": # if the first letter is T
        T_subpop.append(True)
    else:
        T_subpop.append(False)
    print(T_subpop)

Th
[True]
Tc
[True, True]
Treg
[True, True, True]
B
[True, True, True, False]
NK
[True, True, True, False, False]
DC
[True, True, True, False, False, False]


To show you the power of Python, a much more efficient way to write the same algorithm would be with a fancy device called a **list comprehension**, which builds a list on the go — but you don't need to know how to use these to be a good Python coder:

In [315]:
T_subpop = [x[0]=='T' for x in d['cell_type']]
T_subpop

[True, True, True, False, False, False]

An even faster way to do this is to use Pandas' **string methods**. These methods allow us to extremely efficiently apply string functions to columns that contain string data. String methods are accessible by calling `DataFrame[col].str.{insert function name}()`.

In our case, we can just use the function encoded by square brackets `[]` to access the first character:

In [316]:
d['cell_type'].str[0]

cluster1    T
cluster2    T
cluster3    T
cluster4    B
cluster5    N
cluster6    D
Name: cell_type, dtype: str

And check if it's equal to "T" and add it to the DataFrame:

In [317]:
d['cell_type'].str[0]=='T'

cluster1     True
cluster2     True
cluster3     True
cluster4    False
cluster5    False
cluster6    False
Name: cell_type, dtype: bool

In [318]:
d['T_subpop'] = d['cell_type'].str[0]=='T'
d

,cell_type,cell_count,Cell %,is_count_large,T_subpop
cluster1,Th,123,5.118602,False,True
cluster2,Tc,234,9.737828,False,True
cluster3,Treg,345,14.357054,False,True
cluster4,B,456,18.976280,True,False
cluster5,NK,567,23.595506,True,False
cluster6,DC,678,28.214732,True,False


Some other basic string methods:

In [319]:
d['cell_type'].str.lower()

cluster1      th
cluster2      tc
cluster3    treg
cluster4       b
cluster5      nk
cluster6      dc
Name: cell_type, dtype: str

In [320]:
d['cell_type'].str.upper()

cluster1      TH
cluster2      TC
cluster3    TREG
cluster4       B
cluster5      NK
cluster6      DC
Name: cell_type, dtype: str

In [321]:
d['cell_type'].str.len() # returns the length of each string

cluster1    2
cluster2    2
cluster3    4
cluster4    1
cluster5    2
cluster6    2
Name: cell_type, dtype: int64

We can also concatenate all the strings in a columns:

In [322]:
print(d['cell_type'].str.cat())
print(d['cell_type'].str.cat(sep=", "))

ThTcTregBNKDC
Th, Tc, Treg, B, NK, DC


Or concatenate each string with a string we define:

In [323]:
d['cell_type'].str.cat(['_1', '_2', '_3', '_4', '_5', '_6'])

cluster1      Th_1
cluster2      Tc_2
cluster3    Treg_3
cluster4       B_4
cluster5      NK_5
cluster6      DC_6
Name: cell_type, dtype: str

We can even find and replace strings within the column using regular expressions:

In [324]:
d['cell_type'].str.replace("T", "T_")

cluster1      T_h
cluster2      T_c
cluster3    T_reg
cluster4        B
cluster5       NK
cluster6       DC
Name: cell_type, dtype: str

We can even split strings across columns!

In [325]:
d['well_id'] = ['A_1', 'A_2', 'A_3', 'B_1', 'B_2', 'B_3']
d['well_id'].str.split('_')

cluster1    [A, 1]
cluster2    [A, 2]
cluster3    [A, 3]
cluster4    [B, 1]
cluster5    [B, 2]
cluster6    [B, 3]
Name: well_id, dtype: object

Which returns a Series of lists! If we wanted to split the column into two columns, we could use the `expand` argument:

In [326]:
d['well_id'].str.split('_', expand=True)

,0,1
cluster1,A,1
cluster2,A,2
cluster3,A,3
cluster4,B,1
cluster5,B,2
cluster6,B,3


What if we wanted to directly add the split columns to our DataFrame without keeping their boring column names (`0` and `1`)? It's a bit like magic: simply assign the split DataFrame to new columns in the original DataFrame! (Pandas will copy the values only, not the old column names.)

In [327]:
d[['well_row', 'well_col']] = d['well_id'].str.split('_', expand=True)
d

,cell_type,cell_count,Cell %,is_count_large,T_subpop,well_id,well_row,well_col
cluster1,Th,123,5.118602,False,True,A_1,A,1
cluster2,Tc,234,9.737828,False,True,A_2,A,2
cluster3,Treg,345,14.357054,False,True,A_3,A,3
cluster4,B,456,18.976280,True,False,B_1,B,1
cluster5,NK,567,23.595506,True,False,B_2,B,2
cluster6,DC,678,28.214732,True,False,B_3,B,3


And so on. The possibilities are almost endless, and if there isn't a function that does exactly what you want, odds are you can neatly combine a few existing functions to do it!

## Categorical data

We've seen multiple types of data so far: integers, floats (decimals), booleans (`True` and `False`), strings, and even lists. Categorical data is just another type of data.

Say we had a variable that is one of a limited and defined set of potential values, like cell type, treatment protocol, patient ID, etc. One way to store that data would be into strings: `"Treatment"` vs `"Placebo"`, `"Patient 01"` vs `"Patient 02"`, `"CD4 Th"` vs `"cDC1"`. But when you have tens of thousands of samples, this can take up a lot of computer memory and thus slow down operations. Another, much more efficient way to store this information would be by representing each category with an integer, which would take a lot less memory space than a string, and then keep on the side a simple map that tells us which integer maps to which category. Something like `0: treatment, 1: placebo`. Then, Python would only have to track the small and light integers rather than the hefty strings. This is **categorical data** where the `categories` are `treatment` and `placebo`.

Pandas handles categorical data very seamlessly. When viewing categorical data, you don't actually see the internal representation and you don't need to worry about the mapping (Pandas does it for you!). Superficially, the data looks the same; it's just organized differently on the back-end. 

But categorical data is not just about saving space and speeding up computations — although these are already great justifiers to use it. Many Python statistical libraries and functions rely on categorical data!

For those of you familiar with R, categorical data is equivalent to R's `factor` and `levels`.

Creating Series and DataFrames with categorical data is easy: you just need to set the `dtype="category"`):

In [328]:
s = pd.Series(['T', 'T', 'T', 'NK', 'NK', 'B', 'B'], dtype="category")
s

0     T
1     T
2     T
3    NK
4    NK
5     B
6     B
dtype: category
Categories (3, str): ['B', 'NK', 'T']

Notice the last two lines. Pandas automatically detected the categories and ordered them alphabetically (though you can order them however you want).

If you wanted to change an element in the data, make sure the new value is one of the `categories`:

In [329]:
s[0] = 'NK'
s

0    NK
1     T
2     T
3    NK
4    NK
5     B
6     B
dtype: category
Categories (3, str): ['B', 'NK', 'T']

Because if it isn't, you'll get an error:

In [330]:
s[0] = 'DC' # expect an error!!!

TypeError: Cannot setitem on a Categorical with a new category (DC), set the categories first

You can change categories by first accessing the categorical data methods via `.cat` (just like you would access string methods with `.str`!), and then calling the function you want, like `add_categories()`, `remove_categories()`, `rename_categories()`, or `set_caegories()`. (Note that these functions usually return a copy, so don't forget to reassign the treturned copy to the original variable!)

In [331]:
s = s.cat.add_categories(['DC'])
s

0    NK
1     T
2     T
3    NK
4    NK
5     B
6     B
dtype: category
Categories (4, str): ['B', 'NK', 'T', 'DC']

In [332]:
s[0] = 'DC'
s

0    DC
1     T
2     T
3    NK
4    NK
5     B
6     B
dtype: category
Categories (4, str): ['B', 'NK', 'T', 'DC']

If you remove a category, though, something weird happens:

In [333]:
s.cat.remove_categories(['B'])

0     DC
1      T
2      T
3     NK
4     NK
5    NaN
6    NaN
dtype: category
Categories (3, str): ['DC', 'NK', 'T']

`NaN` stands for "not a number". This is a symbol defined by IEEE standards to represent missing, invalid, or undefined values. We will see it again when we explore missing data.

One of the most realistic and important ways to create categorical data is by converting existing data (that you likely imported from a CSV file) into categorical data. Thankfully, that is very easily done with `.astype()` with the argument `"category"`. 

In [334]:
d['cell_type'].astype('category')

cluster1      Th
cluster2      Tc
cluster3    Treg
cluster4       B
cluster5      NK
cluster6      DC
Name: cell_type, dtype: category
Categories (6, str): ['B', 'DC', 'NK', 'Tc', 'Th', 'Treg']

Like most of the functions we have explored, this one also returns a modified copy of the original variable without actually modifying the original variable. To make that change permanent, use reassignment:

In [335]:
d['cell_type'] = d['cell_type'].astype('category')
d

,cell_type,cell_count,Cell %,is_count_large,T_subpop,well_id,well_row,well_col
cluster1,Th,123,5.118602,False,True,A_1,A,1
cluster2,Tc,234,9.737828,False,True,A_2,A,2
cluster3,Treg,345,14.357054,False,True,A_3,A,3
cluster4,B,456,18.976280,True,False,B_1,B,1
cluster5,NK,567,23.595506,True,False,B_2,B,2
cluster6,DC,678,28.214732,True,False,B_3,B,3


In [336]:
d.dtypes

cell_type         category
cell_count           int64
Cell %             float64
is_count_large        bool
T_subpop              bool
well_id                str
well_row               str
well_col               str
dtype: object

`astype()` can also be used to convert to other types, including converting `category` back to `str` to regain the original data:

In [337]:
d['cell_type'].astype("str")

cluster1      Th
cluster2      Tc
cluster3    Treg
cluster4       B
cluster5      NK
cluster6      DC
Name: cell_type, dtype: str

## Missing data

Pandas is designed to deal with real-world data, and missing data is an inevitable part of real-world data, thus Pandas is desiged to deal with missing data! As we saw earlier, the standard symbol for missing, invalid, or undefined data is `NaN` (not a number, although it can also represent missing text and values of other types), although Pandas also uses `NaT` (not a time), `None`, and `NA` in other settings.

To play around with missing data, let's bring back an old friend:

In [338]:
d2

,gene1,gene2,gene3
0,1,1,1
1,2,2,2
2,3,3,3
3,4,4,4
4,5,5,5


To insert missing data, we can use `None`:

In [339]:
d2.iloc[1, [0,2]] = None
d2.iloc[4,0] = None
d2

,gene1,gene2,gene3
0,1.0,1,1.0
1,NaN,2,NaN
2,3.0,3,3.0
3,4.0,4,4.0
4,NaN,5,5.0


### Calculations with missing data
Arithmetic operations between Pandas objects usually **propagate** missing values, meaning that a calculation that includes missing values will return a missing value:

In [340]:
d2 + 10

,gene1,gene2,gene3
0,11.0,11,11.0
1,NaN,12,NaN
2,13.0,13,13.0
3,14.0,14,14.0
4,NaN,15,15.0


However, that is not always the case:

In [341]:
d2.sum(axis=1) # sum along the rows

0     3.0
1     2.0
2     9.0
3    12.0
4    10.0
dtype: float64

Here, the sum clearly proceeds despite the missing values, which take a value `0`.

In [342]:
d2.prod(axis=1)

0     1.0
1     2.0
2    27.0
3    64.0
4    25.0
dtype: float64

The product also proceeds as usual, with missing values taking on a value of `1`.

However, if we want to preserve and propagate missing values, we can set the `skipna` argument to `False`:

In [343]:
d2.sum(axis=1, skipna=False)

0     3.0
1     NaN
2     9.0
3    12.0
4     NaN
dtype: float64

In [344]:
d2.prod(axis=1, skipna=False)

0     1.0
1     NaN
2    27.0
3    64.0
4     NaN
dtype: float64

### Finding and dropping missing data

To find missing data, we can use `isna()`:

In [345]:
d2.isna()

,gene1,gene2,gene3
0,False,False,False
1,True,False,True
2,False,False,False
3,False,False,False
4,True,False,False


If the DataFrame is so large we can't see every single value, we can additionally use `any()` or `all()`, which return a boolean for whether any or all values are `True`. Both of these functions take an `axis` argument which determines wether the any/all assessment should be done along the columns (`0`), rows (`1`), or entire DataFrame (`None`).

In [346]:
d2.isna().any(axis=0) # trying setting axis to either 0, 1, or None

gene1     True
gene2    False
gene3     True
dtype: bool

Notice how we used `any()` right after calling `isna()`! This is called **method chaining**, and it's an essential and extremely convenient feature of Pandas. Functions are executed in the order they are called, and each function must return an object that compatible with the next function (i.e. an object that has the next function as a method). In this case, `isna()` returns a `DataFrame` object, which indeed does have `any()` and `all()` as methods. (You can double check by running `dir(d2)` and `dir(d2.isna())`!)

`notna()` is the complement of `isna()`, as it returns a bool indicating whether a value is **not** not-a-number. In other terms, it indicates whether a value is defined. 

In [347]:
d2.notna()

,gene1,gene2,gene3
0,True,True,True
1,False,True,False
2,True,True,True
3,True,True,True
4,False,True,True


In [348]:
d2.notna().all(axis=0)

gene1    False
gene2     True
gene3    False
dtype: bool

To drop missing data, we can use `dropna()`, which also takes an `axis` argument:

In [349]:
d2.dropna(axis=0) # drops rows with missing data

,gene1,gene2,gene3
0,1.0,1,1.0
2,3.0,3,3.0
3,4.0,4,4.0


In [350]:
d2.dropna(axis=1) # drops columns with missing data

,gene2
0,1
1,2
2,3
3,4
4,5


`dropna()` can get a little fancier: argument `how` can be used to determine whether rows/columns should be dropped if they have `any`/`all` of their values missing; `thresh` allows us to set a threshold of missing values above which a row/column is dropped; and `subset` enables us to select which columns/rows to consider when dropping rows/columns.

### Filling missing values

To fill missing values, we can use `fillna()`:

In [351]:
d2.fillna(1000)

,gene1,gene2,gene3
0,1.0,1,1.0
1,1000.0,2,1000.0
2,3.0,3,3.0
3,4.0,4,4.0
4,1000.0,5,5.0


Or we could use forward- or backward-filling, meaning the missing values take on the values that precede or follow them. This can be done with `ffill()` for forward filling and `bfill()` for backward filling:

In [352]:
d2.ffill()

,gene1,gene2,gene3
0,1.0,1,1.0
1,1.0,2,1.0
2,3.0,3,3.0
3,4.0,4,4.0
4,4.0,5,5.0


In [353]:
d2.bfill()

,gene1,gene2,gene3
0,1.0,1,1.0
1,3.0,2,3.0
2,3.0,3,3.0
3,4.0,4,4.0
4,NaN,5,5.0


(Notice how that NaN in the last row isn't filled because there is no following row whose value it can copy!)

Another option would be to fill using a `Series` object with labels that correspond to the labels of the DataFrame. For instance, if we wanted to fill each missing value with the mean of its column, we can directly use the output of `mean()` (`fillna` sees the labels returned by `mean()` and perfectly matches them to the columns of the DataFrame.)

In [354]:
d2.fillna(d2.mean())

,gene1,gene2,gene3
0,1.000000,1,1.00
1,2.666667,2,3.25
2,3.000000,3,3.00
3,4.000000,4,4.00
4,2.666667,5,5.00


Pandas also provides a more sophisticated set of tools for interpolation through the function `interpolate()`, which unless otherwise specified uses linear interpolation, meaning it assumes values along the column are equally spaced:

In [355]:
d2.interpolate()

,gene1,gene2,gene3
0,1.0,1,1.0
1,2.0,2,2.0
2,3.0,3,3.0
3,4.0,4,4.0
4,4.0,5,5.0


It works almost perfectly! Because there is no value below that last NaN, it assumes the value of the row that precedes it directly (in this case, 4). There are many other interpolation algorithms that Pandas offers through this wondrous function. Check out the documentation. Have fun.

## Duplicate data

If you play around with creating DataFrames, you might notice that Pandas mysteriously allows duplicate columns and indexes. This is because Pandas is built to deal with real-world data, and real-world data may contain duplicate values. It doesn't mean that Pandas condones duplicates — there will often be negative consequences to having duplicate columns and rows!

Let's make yet another toy DataFrame:

In [356]:
dup = pd.DataFrame({
    'cell_type': ['T', 'T', 'T', 'B', 'B', 'NK', 'NK'],
    'cell_count': [1000, 1000, 780, 550, 330, 210, 120],
    'sample_id': ['P1', 'P1', 'P2', 'P3', 'P3', 'P2', 'P4'],
}, index = [1,1,1,3,4,5,6])

dup

,cell_type,cell_count,sample_id
1,T,1000,P1
1,T,1000,P1
1,T,780,P2
3,B,550,P3
4,B,330,P3
5,NK,210,P2
6,NK,120,P4


First, it's important to check whether the indexes and columns are unique:

In [357]:
print(dup.index.is_unique)
print(dup.columns.is_unique)

False
True


To check which ones are duplicated, `duplicated()` is your friend. This function returns a boolean array indicating whether a value is duplicated:

In [358]:
dup.index.duplicated()

array([False,  True,  True, False, False, False, False])

Notice that only the second and following duplicates of a value are classified as duplicates by `duplicated()`, although that can be changed via the arguemnt `keep` (check out the docs!):

In [359]:
dup.index.duplicated(keep='last') # try setting keep=False

array([ True,  True, False, False, False, False, False])

To check what values are actually duplicated, simply use the brackets `[]`:

In [360]:
dup.index[dup.index.duplicated()]

Index([1, 1], dtype='int64')

`duplicated()` also applies to columns, not just the index and columns. When no `subset` of columns is selected, it looks at the entire row and evaluates whether it is duplicated or not (the index doesn't matter here):

In [361]:
dup.duplicated()

1    False
1     True
1    False
3    False
4    False
5    False
6    False
dtype: bool

But we can also select a `subset` of columns (one or more) to determine duplication:

In [362]:
dup.duplicated(subset='cell_type')

1    False
1     True
1     True
3    False
4     True
5    False
6     True
dtype: bool

In [363]:
dup.duplicated(subset='sample_id')

1    False
1     True
1    False
3    False
4     True
5     True
6    False
dtype: bool

In [364]:
dup.duplicated(subset=['cell_type', 'sample_id'])

1    False
1     True
1    False
3    False
4     True
5    False
6    False
dtype: bool

And naturally, we can use indexing to look at the content of the duplicated rows:

In [365]:
dup[dup.duplicated(subset=['cell_type', 'sample_id'], keep=False)] # keep=False shows all duplicates,including the og one

,cell_type,cell_count,sample_id
1,T,1000,P1
1,T,1000,P1
3,B,550,P3
4,B,330,P3


Dropping duplicates is just as easy: it is done with `drop_duplicates()`, which can also take `subset` and `keep` as arguments to determine which columns to look at and which duplicates to drop.

In [366]:
dup.drop_duplicates(subset=dup.columns, keep="first") # try changing the subset and keep arguments!

,cell_type,cell_count,sample_id
1,T,1000,P1
1,T,780,P2
3,B,550,P3
4,B,330,P3
5,NK,210,P2
6,NK,120,P4


## Concatenating DataFrames

What if we had two data frames that we wanted to merge somehow? There are so many ways to merge data sets, and Pandas provides a very flexible, convenient, and intuitive framework to do so.

First, let's create some more toy DataFrames to illustrate this framework:

In [367]:
df1 = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2", "A3"],
        "B": ["B0", "B1", "B2", "B3"],
        "C": ["C0", "C1", "C2", "C3"],
        "D": ["D0", "D1", "D2", "D3"],
    },
    index = [0, 1, 2, 3]
)
df2 = pd.DataFrame(
    {
        "A": ["A4", "A5", "A6"],
        "B": ["B4", "B5", "B6"],
        "C": ["C4", "C5", "C6"],
        "D": ["D4", "D5", "D6"],
    },
    index = [3,4,5]
)
df3 = pd.DataFrame(
    {
        "A": ["A7", "A8", "A9"],
        "B": ["B7", "B8", "B9"],
        "C": ["C7", "C8", "C9"],
        "D": ["D7", "D8", "D9"],
    },
    index = [3,5,6]
)

`concat()` takes an arbitrarily long list of DataFrames and concatenates them along an axis of your choice (0 for rows, 1 for columns).

In [368]:
pd.concat([df1, df2, df3], ignore_index=False) # try setting ignore_index=True

,A,B,C,D
0,A0,B0,C0,D0
1,A1,B1,C1,D1
2,A2,B2,C2,D2
3,A3,B3,C3,D3
3,A4,B4,C4,D4
4,A5,B5,C5,D5
5,A6,B6,C6,D6
3,A7,B7,C7,D7
5,A8,B8,C8,D8
6,A9,B9,C9,D9


(Notice how the index duplicates are allowed!)

What if one of the concatenated DataFrames did not have all the columns shared by the other DataFrames? This is where `join` comes in. `join` dictates the logic of inclusion along the axis that is not being concatenated (columns if we're concatenating along the rows). `join="inner"` means that only the overlapping columns will be included (the intersection), whereas `join="outer"` means that all columns will be included (the union). By default, `join` is set to `outer` so all columns are included. In the example below, `df4` has column `X` instead of `D`:
- `outer` means columns `ABCDX` will be included, and missing values will be indicated as `NaN`
- `inner` means only columns `ABC` will be included in the final concatenat

In [369]:
df4 = pd.DataFrame(
    {
        "A": ["A7", "A8"],
        "B": ["B7", "B8"],
        "C": ["C7", "C8"],
        "X": ["X7", "X8"],
    },
    index = [5, 6]
)

In [370]:
pd.concat([df1, df2, df4], join="outer") # try join=inner vs outer

,A,B,C,D,X
0,A0,B0,C0,D0,NaN
1,A1,B1,C1,D1,NaN
2,A2,B2,C2,D2,NaN
3,A3,B3,C3,D3,NaN
3,A4,B4,C4,D4,NaN
4,A5,B5,C5,D5,NaN
5,A6,B6,C6,D6,NaN
5,A7,B7,C7,NaN,X7
6,A8,B8,C8,NaN,X8


(Notice the NaNs in the mismatched columns!)

What if we wanted to concatenated along the columns? It's the exact same logic as concatenating along the rows, except it's along the columns!

In [371]:
pd.concat([df1, df2, df3], axis=1, join='outer', ignore_index=False) # try join=inner

,A,B,C,D,A,B,C,D,A,B,C,D
0,A0,B0,C0,D0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A1,B1,C1,D1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A2,B2,C2,D2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A3,B3,C3,D3,A4,B4,C4,D4,A7,B7,C7,D7
4,NaN,NaN,NaN,NaN,A5,B5,C5,D5,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,A6,B6,C6,D6,A8,B8,C8,D8
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A9,B9,C9,D9


Here is a very helpful diagram I found on StackOverflow:

<center><img src="figures/concat_join.jpg"></center>


## Merging DataFrames

What if we had two tables we wanted to merge according to a shared column? For instance, we say we have two tables offering different kinds of information about overlapping sets of genes, and we want to merge them into one table that has all the information in one place. 

<center><img src="figures/08_merge_left.svg"></center>


Pandas answer to this common situation is `merge()`, a function which takes `left` and `right` DataFrames and merges them `on` based on one or more key columns, using a merging logic defined by a `how` argument:
- `left`: keep only the keys in the left DataFrame
- `right`: keep only the keys in the right DataFrame
- `inner`: keep only the keys that are shared between the two DataFrames
- `outer`: keep the union of all keys
- `cross`: generate a Cartesian product of the keys
- `right_anti`: only the right DataFrame keys which are not in the left DataFrame
- `left_anti`: only the left DataFrame keys which are not in the right DataFrame

Let's make some toys:


In [372]:
left = pd.DataFrame(
    {
        "key": ['k0', 'k1', 'k2', 'k3'],
        "A": ['A0', 'A1', 'A2', 'A3'],
        "B": ['B0', 'B1', 'B2', 'B3']
    }
)
right = pd.DataFrame(
    {
        "key": ['k0', 'k1', 'k2', 'k4', 'k5', 'k2'],
        "C": ['C0', 'C1', 'C2', 'C4', 'C5', 'C2*'],
        "D": ['D0', 'D1', 'D2', 'D4', 'D5', 'D2*']
    }
)

In [373]:
right

,key,C,D
0,k0,C0,D0
1,k1,C1,D1
2,k2,C2,D2
3,k4,C4,D4
4,k5,C5,D5
5,k2,C2*,D2*


In [374]:
pd.merge(left, right, on='key', how='left') # have fun with the how argument!

,key,A,B,C,D
0,k0,A0,B0,C0,D0
1,k1,A1,B1,C1,D1
2,k2,A2,B2,C2,D2
3,k2,A2,B2,C2*,D2*
4,k3,A3,B3,NaN,NaN


Notice that if a key is duplicated in one of the DataFrames, it will also be duplicated in the merge. Here, `k2` is duplicated in `right`, so a merge that includes `k2` will have two rows for `k2` data.

You can even keep track of where a key came from using `indicator`, which will create a new column named `_merge` containing that information:

In [375]:
pd.merge(left, right, on='key', how='outer', indicator=True)

,key,A,B,C,D,_merge
0,k0,A0,B0,C0,D0,both
1,k1,A1,B1,C1,D1,both
2,k2,A2,B2,C2,D2,both
3,k2,A2,B2,C2*,D2*,both
4,k3,A3,B3,NaN,NaN,left_only
5,k4,NaN,NaN,C4,D4,right_only
6,k5,NaN,NaN,C5,D5,right_only


If you read the documentation, you might stumble upon `DataFrame.join()`, which is very similar to `merge()`, except it primarily merges based on the index of at least one of the two DataFrames. `merge()` is more generalizable, as it can deal with indexes and columns in either one of the merged DataFrames.  

## Long vs. wide data formats

Say we have some gene expression information from bulk RNA sequencing data that includes multiple samples. One way in which that data is formatted is the following, in which each gene-sample pair takes up a row:

| gene | sample | gene_expression |
| :-- | :-- | :-- |
| cd4 | P01 | 11|
| cd4 | P02 | 12|
| cd4 | P03 | 13|
| cd8 | P01 | 21|
| cd8 | P02 | 22|
| cd8 | P03 | 23|
| ccr7 | P01 | 31|
| ccr7 | P02 | 32|
| ccr7 | P03 | 33|

This looks okay, but here's another way that many would argue is more intuitive and less redundant:

| gene | P01 | P02 | P03 |
| :-- | :-- | :-- | :-- |
| cd4 | 11 | 12 | 13 |
| cd8 | 21 | 22 | 23 | 
| ccr7 | 31 | 32 | 33|

These two ways of organizing data are so common that they have their own names: the first is called the **long format**, and the second is called the **wide format**. In the long format, each row contains the independent and dependent (measured) variables for one single data point. In the wide format, the columns and the rows usually represent the independent variables (i.e. sample ID, gene name, or date), while the values in the middle are the measured ones. 

The wide format, perhaps because it is less redundant and brings the measureed values closer to each other, tends to be the more human-readable one. However, some machines and algorithms actually tend to prefer the long format. This is because reading a row is a lot less computationally costly than reading a column, and most data formats are stored row-by-row. The main factors that determines which format you pick is simply the software that you are using and the downstream applications. 

To switch between the long and wide formats is to **pivot** a table. Pandas offers very intuitive functions to do just that: `pivot()`, which goes from long to wide, and `melt()`, which takes a wide table and "melts" it into a long format one. There are of course many more slightly different functions, but these are the main two that you should know about. 

To `pivot()` a table from long to wide, you need three things: 
- `index`: the column which will organize the rows (`gene` in the example above)
- `columns`: the column which will provide the column names (`sample` above)
- `values`: the column which will provide the values with which to fill the table (`gene_expression` above)

Let's try it out:

In [376]:
['cd4']*3 + ['cd4']*3

['cd4', 'cd4', 'cd4', 'cd4', 'cd4', 'cd4']

In [377]:
exp = pd.DataFrame({
    "gene": ['cd4']*3 + ['cd8']*3 + ['ccr7']*3,
    "sample": ['P01', 'P02', 'P03'] * 3,
    "gene_expression": [11,12,13,21,22,23,31,32,33],
})
exp

,gene,sample,gene_expression
0,cd4,P01,11
1,cd4,P02,12
2,cd4,P03,13
3,cd8,P01,21
4,cd8,P02,22
5,cd8,P03,23
6,ccr7,P01,31
7,ccr7,P02,32
8,ccr7,P03,33


In [378]:
exp_wide = exp.pivot(index="gene", columns="sample", values="gene_expression")
exp_wide

sample,P01,P02,P03
gene,,,
ccr7,31,32,33
cd4,11,12,13
cd8,21,22,23


Now let's `melt()` our wide-format DataFrame. `melt()` takes the following arguments:
- `id_vars`: the columns that define the identifier variables (aka indenpendent variables), these columns are preserved, and each row is repeated for every variable (as expected for long-format data)
- `value_var`: the columns that define the values (default is all the columns that are not in `id_vars`)
- `var_name`: the name given to the variable column (default is `"variable"`)
- `value_name`: the name given to the values column (default is `"value"`)

<center><img src="figures/reshaping_melt.png"></center>

One important detail is that `melt()` does not take the index as an `id_vars`, which is a problem because the index (`gene`) is the identifier variable in our wide-format table above. The solution is simple, first turn it into a normal column using `reset_index()`, and then use `melt()`!

In [379]:
exp_wide.reset_index().melt(id_vars='gene', value_name='gene_expression')

,gene,sample,gene_expression
0,ccr7,P01,31
1,cd4,P01,11
2,cd8,P01,21
3,ccr7,P02,32
4,cd4,P02,12
5,cd8,P02,22
6,ccr7,P03,33
7,cd4,P03,13
8,cd8,P03,23


## Reading data into Pandas

Enough with the toy data sets! It is about time we import some real, serious data so we can truly see the power of Pandas. 

Pandas can read and import data in many different types of formats, including CSV (comma-separated values, or any values separated by a pre-defined delimiter), XLSX (Excel), HTML, Parquet, HDF5, JSON, GBQ, SQL, and many others! Here's a helpful schematic courtesy of the Pandas docs:

<center><img src="figures/02_io_readwrite.svg"></center>

Indeed, reading data into Pandas is as simply as calling the function `read_[insert data type]([insert data location])`. Let's try it with our very first toy dataset: `titanic.

In [4]:
df = pd.read_csv('titanic.csv')
print(df.shape)
df

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


Wow! 891 rows and 15 columns! That's nothing for Pandas. 

Note, however, that sometimes you may have to specify important parameters before reading data. For instance, not all data comes in a comma-separated (CSV) format; some data sets may have been saved in a tab- or even space-separated format, so you may have to specify what delimiter was used using the `sep` argument. You may also have to indicate where the column names are using `header`. Another useful argument is `usecols` which allows you to specify which columns you wish to use in your analysis, which could significantly increase performance if you only need to deal with 2 columns out of 100:

In [5]:
pd.read_csv('titanic.csv', usecols=['survived', 'sex', 'age'])

,survived,sex,age
0,0,male,22.0
1,1,female,38.0
2,1,female,26.0
3,1,female,35.0
4,0,male,35.0
...,...,...,...
886,0,male,27.0
887,1,female,19.0
888,0,female,NaN
889,1,male,26.0


Now, let's explore the full data a little:

In [6]:
df.head(10) # try tail!

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


In [7]:
df.fare.describe() # if column name is simple text with underscores, you don't need quotes!

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: fare, dtype: float64

In [8]:
df.age.describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: age, dtype: float64

Easy to count the number of survivors or the number of passengers who traveled alone; we can just `sum()` their respective columns:

In [9]:
df.survived.sum()

np.int64(342)

In [10]:
df.alone.sum()

np.int64(537)

But what if we wanted to count the number of males vs. females on the Titanic? Worry not, as `value_counts()` got you covered:

In [11]:
df.sex.value_counts()

sex
male      577
female    314
Name: count, dtype: int64

In [12]:
df['class'].value_counts()

class
Third     491
First     216
Second    184
Name: count, dtype: int64

In [13]:
df.embark_town.value_counts()

embark_town
Southampton    644
Cherbourg      168
Queenstown      77
Name: count, dtype: int64

## Statistics by category

What if we wanted to compute a statistic — but per group of samples defined by a category instead of aggregated over all data? This is also known as the **split-apply-combine** paradigm: split the data according to the categories in a column, apply the same computations on each split, and then combine the resulting statistics into one table.

<center><img src="figures/06_groupby.svg"></center>

For example, using the data above, what if we wanted to compute the mean age of male vs female passengers, or survival rate of each boarding class? One way to go about this would be to manually select the rows of interest and then compute the statistics:

In [14]:
print(df[df['class']=='First']['survived'].mean())
print(df[df['class']=='Second']['survived'].mean())
print(df[df['class']=='Third']['survived'].mean())

0.6296296296296297
0.47282608695652173
0.24236252545824846


Of course, it already seems like boarding class was a big factor at determining who survived. The world is egregiously unfair, and we are clearly *not* all on the same boat.

Going back to calculating statistics for each category in a column, another way to do so would be the to use `groupby()`, which does the splitting part of the split-apply-combine, the other (apply-combine) steps are automatically performed by the statistical functions we've been using, like `.mean()` and so on.

In [15]:
df[['class', 'survived']].groupby('class').mean()

,survived
class,
First,0.629630
Second,0.472826
Third,0.242363


Notice that we have to first select only the columns we need for this computation: the column based on which we will split the data (`class` in this case), and the columns based on which we wish to do statistical computations (`survived` in this case). Another way of doing this is to select the statistical columns after calling `groupby()`:

In [16]:
df.groupby('class')['survived'].mean()

class
First     0.629630
Second    0.472826
Third     0.242363
Name: survived, dtype: float64


The main difference is that this method returns a Series, whereas the previous one returns a DataFrame.

`groupby()` can also be used on combinations of categories!

In [17]:
fares = df.groupby(['sex', 'class'])['fare'].median()
fares

sex     class 
female  First     82.66455
        Second    22.00000
        Third     12.47500
male    First     41.26250
        Second    13.00000
        Third      7.92500
Name: fare, dtype: float64

(Of course, women pay extra (almost double!) for the same things. This data really shows how classist and sexist the world was — and still is.)

Going back to Pandas, notice that this result is a Series object — but it has mutlitple columns!? The two named columns are actually a `MultiIndex`, which is essentially many layers of indexes. Indexing on a MultiIndex is very intuitive and can be done in a few different ways:

In [18]:
fares['female']

class
First     82.66455
Second    22.00000
Third     12.47500
Name: fare, dtype: float64

In [19]:
fares['female']['First']

np.float64(82.66454999999999)

In [20]:
fares[('female', 'First')] # remeber the name of this ()-bracketed object? It's a tuple!

np.float64(82.66454999999999)

## Writing files with Pandas

<center><img src="figures/02_io_readwrite.svg"></center>

The schematic above about reading files into Pandas also shows very clearly how easy it is to export data using Pandas: `.to_[insert file type]([insert file name])`:

In [21]:
fares.to_csv('example.csv')

As the schematic also shows, you can export Pandas data using a very wide spectrum of data formats, including Excel (`to_excel()`) among many others. But before you use these functions, or any other functions in Pandas, always make sure to read the docs to prevent bugs and learn more about new and formidable options you didn't know you even needed.  

## Exercises

In this exercise, we will analyze a bulk RNA sequencing data set that has been used in this bootcamp for many years (when it was taught in R). Bulk RNA sequencing is simply the sequencing of mRNA moieties in a tissue that has been homogenized, so contrary to single-cell RNA sequencing, we are looking at the average RNA profile over all the cells in the tissue sample. 

This dataset was generated in 2017 from [this study](https://pubmed.ncbi.nlm.nih.gov/28696309/), which is about the effect of Influenza A infection on the central nervous system. We will first explore the data set and then rearrange it to measure some basic statistics. 

1) Read the `rnaseq.csv` data set with Pandas:

In [3]:
rna = pd.read_csv("rnaseq.csv")

2. View the DataFrame and look at it. How many rows and columns does it have?

In [4]:
rna

,gene,sample,expression,organism,age,sex,infection,strain,time,tissue,mouse,ENTREZID,product,ensembl_gene_id,external_synonym,chromosome_name,gene_biotype,phenotype_description,hsapiens_homolog_associated_gene_name
0,Asl,GSM2545336,1170,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,14,109900,"argininosuccinate lyase, transcript variant X1",ENSMUSG00000025533,2510006M18Rik,5,protein_coding,abnormal circulating amino acid level,ASL
1,Apod,GSM2545336,36194,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,14,11815,"apolipoprotein D, transcript variant 3",ENSMUSG00000022548,NaN,16,protein_coding,abnormal lipid homeostasis,APOD
2,Cyp2d22,GSM2545336,4060,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,14,56448,"cytochrome P450, family 2, subfamily d, polype...",ENSMUSG00000061740,2D22,15,protein_coding,abnormal skin morphology,CYP2D6
3,Klk6,GSM2545336,287,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,14,19144,"kallikrein related-peptidase 6, transcript var...",ENSMUSG00000050063,Bssp,7,protein_coding,abnormal cytokine level,KLK6
4,Fcrls,GSM2545336,85,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,14,80891,"Fc receptor-like S, scavenger receptor, transc...",ENSMUSG00000015852,2810439C17Rik,3,protein_coding,decreased CD8-positive alpha-beta T cell number,FCRL2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32423,Mgst3,GSM2545380,2151,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,19,66447,microsomal glutathione S-transferase 3,ENSMUSG00000026688,2010012L10Rik,1,protein_coding,decreased mean corpuscular volume,MGST3
32424,Lrrc52,GSM2545380,5,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,19,240899,leucine rich repeat containing 52,ENSMUSG00000040485,4930413P14Rik,1,protein_coding,abnormal sperm physiology,LRRC52
32425,Rxrg,GSM2545380,49,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,19,20183,"retinoid X receptor gamma, transcript variant 1",ENSMUSG00000015843,Nr2b3,1,protein_coding,abnormal bone mineralization,RXRG
32426,Lmx1a,GSM2545380,72,Mus musculus,8,Female,InfluenzaA,C57BL/6,8,Cerebellum,19,110648,"LIM homeobox transcription factor 1 alpha, tra...",ENSMUSG00000026686,Lmx1.1,1,protein_coding,abnormal bony labyrinth,LMX1A


3. While we know that this is bulk RNA-seq data, we don't know exactly how that data is structured, so let's explore it. (Even if we did know the structure, it is always good to double check and get to know it even better!)

    Check out the `sample` column; it carries codes that start with `GSM`. That's a sample code from the [NCBI GEO database](https://www.ncbi.nlm.nih.gov/geo/info/overview.html), where a lot of (usually published) studies deposit their data sets. Count how many samples this data set has, and how many rows each sample has. Do the same thing for other relevant columns in the dataset: count the number of occurences of each unique value they countain.

In [10]:
rna['sample'].unique()

<ArrowStringArray>
['GSM2545336', 'GSM2545337', 'GSM2545338', 'GSM2545339', 'GSM2545340',
 'GSM2545341', 'GSM2545342', 'GSM2545343', 'GSM2545344', 'GSM2545345',
 'GSM2545346', 'GSM2545347', 'GSM2545348', 'GSM2545349', 'GSM2545350',
 'GSM2545351', 'GSM2545352', 'GSM2545353', 'GSM2545354', 'GSM2545362',
 'GSM2545363', 'GSM2545380']
Length: 22, dtype: str

In [21]:
rna['sample'].value_counts()

sample
GSM2545336    1474
GSM2545337    1474
GSM2545338    1474
GSM2545339    1474
GSM2545340    1474
GSM2545341    1474
GSM2545342    1474
GSM2545343    1474
GSM2545344    1474
GSM2545345    1474
GSM2545346    1474
GSM2545347    1474
GSM2545348    1474
GSM2545349    1474
GSM2545350    1474
GSM2545351    1474
GSM2545352    1474
GSM2545353    1474
GSM2545354    1474
GSM2545362    1474
GSM2545363    1474
GSM2545380    1474
Name: count, dtype: int64

In [22]:
rna['gene'].value_counts()

gene
Asl        22
Apod       22
Cyp2d22    22
Klk6       22
Fcrls      22
           ..
Mgst3      22
Lrrc52     22
Rxrg       22
Lmx1a      22
Pbx1       22
Name: count, Length: 1474, dtype: int64

Now try to figure out the experimental design of the data set. That is, how are the samples organized in terms of the columns? You may find `groupby()` to be very useful here!

In [18]:
rna.groupby(['infection', 'sex'])['sample'].value_counts()

infection    sex     sample    
InfluenzaA   Female  GSM2545336    1474
                     GSM2545339    1474
                     GSM2545342    1474
                     GSM2545344    1474
                     GSM2545351    1474
                     GSM2545352    1474
                     GSM2545362    1474
                     GSM2545380    1474
             Male    GSM2545340    1474
                     GSM2545341    1474
                     GSM2545345    1474
                     GSM2545346    1474
                     GSM2545347    1474
                     GSM2545350    1474
                     GSM2545363    1474
NonInfected  Female  GSM2545337    1474
                     GSM2545338    1474
                     GSM2545348    1474
                     GSM2545353    1474
             Male    GSM2545343    1474
                     GSM2545349    1474
                     GSM2545354    1474
Name: count, dtype: int64

4. Find the aggregate average of gene expression, and find it for the different subgroups in columns `sex` and `infection`.

In [64]:
rna.groupby('sex')['expression'].mean()

sex
Female    1882.053652
Male      1834.333379
Name: expression, dtype: float64

In [58]:
rna.groupby('infection')['expression'].mean()

infection
InfluenzaA     1847.539620
NonInfected    1887.840473
Name: expression, dtype: float64

5. You realize the average won't be enough. Do the same thing as above but with `describe()` instead; this should return a list of statistics of interest.

In [65]:
rna.groupby('sex')['expression'].describe()

,count,mean,std,min,25%,50%,75%,max
sex,,,,,,,,
Female,17688.0,1882.053652,4618.386463,0.0,72.0,552.0,1926.25,102790.0
Male,14740.0,1834.333379,4346.500134,0.0,61.0,541.0,1946.00,98658.0


In [66]:
rna.groupby('infection')['expression'].describe()

,count,mean,std,min,25%,50%,75%,max
infection,,,,,,,,
InfluenzaA,22110.0,1847.539620,4424.175680,0.0,68.0,545.0,1907.00,91642.0
NonInfected,10318.0,1887.840473,4648.808184,0.0,64.0,554.0,1988.75,102790.0


6. Now use `describe()` again, but after grouping by both `sex` and `infection`.

In [68]:
rna.groupby(['infection', 'sex'])['expression'].describe()

count         mean          std  min   25%    50%  \
infection   sex                                                           
InfluenzaA  Female  11792.0  1896.358548  4590.814439  0.0  78.0  563.0   
            Male    10318.0  1791.746559  4225.223806  0.0  59.0  518.0   
NonInfected Female   5896.0  1853.443860  4673.302845  0.0  62.0  528.5   
            Male     4422.0  1933.702623  4616.075289  0.0  65.0  573.0   

                        75%       max  
infection   sex                        
InfluenzaA  Female  1930.25   89445.0  
            Male    1883.50   91642.0  
NonInfected Female  1922.00  102790.0  
            Male    2062.75   98658.0

7. Now let's look at the actual gene expression data. Calculate the average expression of each gene in both the infected and non-infected conditions.

In [74]:
diff = rna.groupby(['gene', 'infection'])['expression'].mean()
diff

gene      infection  
AI504432  InfluenzaA     1062.266667
          NonInfected    1033.857143
AW046200  InfluenzaA      119.066667
          NonInfected     155.285714
AW551984  InfluenzaA      320.933333
                            ...     
Zranb3    NonInfected     191.142857
Zscan22   InfluenzaA      479.400000
          NonInfected     607.428571
Zw10      InfluenzaA     1518.400000
          NonInfected    1546.285714
Name: expression, Length: 2948, dtype: float64

8. Notice how this is a long format table? Let's turn (pivot) it into a wide format table where the index is the genes, the two columns correspond to the two infection statuses, and the values are the expression data. 

In [75]:
diff = diff.reset_index().pivot(index = 'gene', columns = 'infection', values = 'expression')
diff

infection,InfluenzaA,NonInfected
gene,,
AI504432,1062.266667,1033.857143
AW046200,119.066667,155.285714
AW551984,320.933333,238.000000
Aamp,4819.866667,4602.571429
Abca12,4.200000,5.285714
...,...,...
Zkscan3,1833.533333,2105.571429
Zranb1,7268.200000,6014.000000
Zranb3,194.266667,191.142857


9. Now create a new column that is equal to the difference between the infected and non-infected groups, and sort it in ascending order.

In [79]:
diff['diff'] = diff['InfluenzaA'] - diff['NonInfected']
diff.sort_values(by='diff')

infection,InfluenzaA,NonInfected,diff
gene,,,
Plp1,52247.266667,91103.142857,-38855.876190
Nrep,29059.333333,40059.857143,-11000.523810
Aqp4,16083.066667,21928.000000,-5844.933333
Trf,12430.066667,17892.857143,-5462.790476
Sbk1,21777.466667,26700.428571,-4922.961905
...,...,...,...
Pink1,20077.066667,15454.571429,4622.495238
Ttyh1,27148.133333,21453.571429,5694.561905
Apod,22417.266667,11575.428571,10841.838095
